In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_9' : ['index_9 EGDI.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx','raw_SDG_2026.xlsx']}

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[1]

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    # make first row columns names
    historical_kpi_data_raw.columns = historical_kpi_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

### Format Data

In [3]:
# Simulator Data

In [4]:
# Historical KPI Data
# Index data minor reformatting and filtering
historical_kpi_data = historical_kpi_data_raw.copy()
historical_kpi_data = historical_kpi_data.rename(columns={'Unnamed: 0':'KPI ID','Unnamed: 1':'Series Name','Unnamed: 2':'year_historical_automated',
                                        'year':'year_historical_automated', 'Series name':'Series Name', 'KeyID':'KPI ID'})
historical_kpi_data = historical_kpi_data.iloc[1:]
# - Values to change, match simulator values
replacement_dict = {
    'Mean years of schooling': 'Mean Year of Schooling',
    'Expected years of schooling': 'Expected Year of Schooling'
}
historical_kpi_data['Series Name'] = historical_kpi_data['Series Name'].replace(replacement_dict)
# - long format
historical_kpi_data_long = historical_kpi_data.melt(id_vars=['KPI ID','Series Name','year_historical_automated'], 
                                           var_name='ISO', value_name='data_historical_automated')        # convert wide to long
historical_kpi_data_long = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
# - format values for consistency
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')

series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

In [5]:
simulator_data = simulator_data_raw.copy()
simulator_data = simulator_data.rename(columns={'Unnamed: 0':'KeyID','Unnamed: 1':'Series name','Unnamed: 2':'Year',
                                        'year':'Year'})
simulator_data.columns = simulator_data.iloc[0]
simulator_data = simulator_data[1:].reset_index(drop=True)
new_names = ['Country', 'ISO', 'Region', 'Sub-region', 'Income']
simulator_data.columns = new_names + list(simulator_data.columns[5:])

# Convert to long format
simulator_data = simulator_data.melt(id_vars=new_names, 
                                     var_name='Series Name', value_name='data_report')      # convert wide to long
cols_to_strip = ['Country', 'ISO', 'Series Name']
simulator_data[cols_to_strip] = simulator_data[cols_to_strip].apply(lambda x: x.str.strip())
simulator_data['data_report'] = pd.to_numeric(simulator_data['data_report'], errors='coerce')
simulator_data = simulator_data.drop(columns=['Region', 'Sub-region', 'Income'])
# - Join in KPI ID
simulator_data = simulator_data.merge(series_to_kpi, how='left')

In [6]:
def find_floor_cap(group, data_col='data_report', suffix=''):
    counts = group[data_col].value_counts()
    min_val = group[data_col].min()
    max_val = group[data_col].max()
    min_count = counts.get(min_val, 0)
    max_count = counts.get(max_val, 0)

    is_floor = pd.notna(min_val) and min_count > 4 and float(min_val).is_integer()
    is_cap = pd.notna(max_val) and max_count > 4 and float(max_val).is_integer()

    return pd.Series({
        f'possible_cap{suffix}': int(max_val) if is_cap else pd.NA,
        f'cap_count{suffix}': int(max_count) if is_cap else pd.NA,
        f'possible_floor{suffix}': int(min_val) if is_floor else pd.NA,
        f'floor_count{suffix}': int(min_count) if is_floor else pd.NA,
    })


In [7]:
int_cols = ['possible_cap_sim', 'cap_count_sim', 'possible_floor_sim', 'floor_count_sim']
simulator_cap_floor = (
    simulator_data
    .groupby('KPI ID')
    .apply(lambda g: find_floor_cap(g, data_col='data_report', suffix='_sim'))
    .reset_index()
)
simulator_cap_floor[int_cols] = simulator_cap_floor[int_cols].astype('Int64')

int_cols_hist = ['possible_cap_hist', 'cap_count_hist', 'possible_floor_hist', 'floor_count_hist']
historical_kpi_cap_floor = (
    historical_kpi_data_long
    .groupby('KPI ID')
    .apply(lambda g: find_floor_cap(g, data_col='data_historical_automated', suffix='_hist'))
    .reset_index()
)
historical_kpi_cap_floor[int_cols_hist] = historical_kpi_cap_floor[int_cols_hist].astype('Int64')

In [8]:
def flag_discrepancy(row, col_sim, col_hist):
    val_sim = row[col_sim]
    val_hist = row[col_hist]
    # Only evaluate if BOTH values are available
    if pd.notna(val_sim) and pd.notna(val_hist):
        return 1 if val_sim != val_hist else 0
    return 0  # not comparable -> treated as no discrepancy
    
discrepancy_cap_floor = simulator_cap_floor.merge(historical_kpi_cap_floor)
discrepancy_cap_floor['cap_discrepancy'] = discrepancy_cap_floor.apply(
    lambda row: flag_discrepancy(row, 'possible_cap_sim', 'possible_cap_hist'),
    axis=1)
discrepancy_cap_floor['floor_discrepancy'] = discrepancy_cap_floor.apply(
    lambda row: flag_discrepancy(row, 'possible_floor_sim', 'possible_floor_hist'),
    axis=1)
discrepancy_cap_floor = discrepancy_cap_floor.merge(series_to_kpi, how='left')
cols = ['Series Name'] + [col for col in discrepancy_cap_floor.columns if col != 'Series Name']
discrepancy_cap_floor = discrepancy_cap_floor[cols]

In [57]:
discrepancy_cap_floor.to_clipboard()